In [1]:
!nvidia-smi

Wed Oct  7 22:17:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import numpy as np
import cupy as cp
from numba import cuda

print("NumPy", np.__version__, "| CuPy", cp.__version__)
print("Numba sees a GPU:", cuda.is_available())
print("GPU:", cuda.get_current_device().name)

NumPy 2.1.3 | CuPy 14.0.1
Numba sees a GPU: True
GPU: Tesla T4


In [3]:
import time

rng = np.random.default_rng(seed=0)

def make_matrices(n):
    A = rng.random((n, n), dtype=np.float32)
    B = rng.random((n, n), dtype=np.float32)
    return A, B

In [4]:
# Matrix multiplication (the 3 loops show how this becomes n^3)
def matmul_python(A, B):
    n = len(A)
    C = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            total = 0.0
            for k in range(n):
                total += A[i][k] * B[k][j]
            C[i][j] = total
    return C

In [5]:
A, B = make_matrices(64)
C_python = matmul_python(A.tolist(), B.tolist())
C_numpy = A @ B
print("Matches NumPy:", np.allclose(C_python, C_numpy, rtol=1e-4))

Matches NumPy: True


In [6]:
for n in [32, 64, 128, 256]:
    A, B = make_matrices(n)
    A_list, B_list = A.tolist(), B.tolist()
    start = time.perf_counter()
    matmul_python(A_list, B_list)
    elapsed = time.perf_counter() - start
    print(f"n = {n:4}: {elapsed:.3f} seconds")
    #the test above is helpful in seeing how doubling n results in O(n^3) time

n =   32: 0.002 seconds
n =   64: 0.018 seconds
n =  128: 0.135 seconds
n =  256: 1.762 seconds
